# Phase 0: Environment Setup

**Goal**: Verify Unsloth, PEFT, and DeepEval work before spending credits.

**Time**: ~5 min | **Cost**: ~1-2 credits

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install deepeval datasets

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Model with Unsloth

This is the actual Unsloth API. No wrappers.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)

## 2. Apply LoRA

Unsloth's `get_peft_model` wraps PEFT's LoRA with optimizations.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

# Check trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 3. Prepare Data

Using HuggingFace datasets directly.

In [ ]:
from datasets import Dataset

# Tiny dataset just to verify training works
data = [{"text": tokenizer.apply_chat_template([
    {"role": "user", "content": f"What is {i}+{i}?"},
    {"role": "assistant", "content": str(i*2)}
], tokenize=False)} for i in range(1, 21)]

dataset = Dataset.from_list(data)
print(f"Samples: {len(dataset)}")
print(f"Example:\n{dataset[0]['text'][:200]}...")

## 4. Train (10 steps)

Using TRL's SFTTrainer directly.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./test_output",
        max_steps=10,
        per_device_train_batch_size=2,
        logging_steps=5,
        optim="adamw_8bit",
        bf16=True,
        report_to="none",
    ),
    max_seq_length=512,
)

trainer.train()

## 5. Test Inference

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is 5+5?"}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=32, temperature=0.1)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. Verify DeepEval

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

print("DeepEval imported successfully!")
print("Note: Full evaluation needs OpenAI API key (we'll set up in Phase 1)")

## ✓ Done

Environment verified. Ready for Phase 1.

In [ ]:
!rm -rf ./test_output